In [ ]:
!pip install roboflow

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
RF_TOKEN = user_secrets.get_secret("RF_TOKEN")

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=RF_TOKEN)
project = rf.workspace("skripshit-2zmum").project("movingcamo")
version = project.version(2)
dataset = version.download("yolov12")

In [ ]:
!pip install ultralytics

In [ ]:
!pip install -q git+https://github.com/sunsmarterjie/yolov12.git roboflow supervision flash-attn

In [ ]:
#!pip install ray==2.40.0

In [ ]:
#!pip install -U ultralytics

In [ ]:
'''dataset_location = '/kaggle/working/MovingCamo-1'
!sed -i '$d' {dataset_location}/data.yaml
!sed -i '$d' {dataset_location}/data.yaml
!sed -i '$d' {dataset_location}/data.yaml
!sed -i '$d' {dataset_location}/data.yaml
!echo -e "test: ../test/images\ntrain: ../train/images\nval: ../valid/images" >> {dataset_location}/data.yaml'''

In [ ]:
!wget https://github.com/sunsmarterjie/yolov12/releases/download/turbo/yolov12s.pt

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov12s.pt')
results = model.train(data="/kaggle/working/MovingCamo-2/data.yaml",
                      optimizer='SGD',
                      lr0=0.01,
                      lrf=0.01,
                      batch=16,
                      momentum=0.937,
                      weight_decay=0.0005,
                      patience=20
                     )

In [ ]:
!zip -r result_3.zip /kaggle/working/runs/detect/train

In [ ]:
from IPython.display import FileLink
FileLink(r'result_3.zip')

In [ ]:
# Evaluate the model
metrics = model.val(data='/kaggle/working/MovingCamo-2/data.yaml')

# Print the evaluation metrics
print(metrics)

# Access specific metrics
print(f"mAP@0.5: {metrics.box.map50}")
print(f"mAP@0.5:0.95: {metrics.box.map}")

# Visualize the results using the plot method
results = model.val(data='/kaggle/working/MovingCamo-2/data.yaml',plots=True)

In [ ]:
!wget 'https://images.fineartamerica.com/images-medium-large-5/1-arctic-wolf-paul-sawerflpa.jpg' -O 'camo_1.jpg'

In [ ]:
import cv2
import matplotlib.pyplot as plt


# Load the trained model. Replace with the actual path to your trained weights.
model_path = '/kaggle/working/runs/detect/train/weights/best.pt' # Update with the correct path to your trained weights
model = YOLO(model_path)

# Testing the Model
test_path = '/kaggle/working/camo_1.jpg'
results = model.predict(source=test_path, save=True, conf=0.25)

# Print the bounding boxes, confidence scores, and class names.
for r in results:
    print(r.boxes) # Print bounding box information
    print(r.boxes.conf) # Print confidence scores
    print(r.boxes.cls) # Print predicted classes

# Display the image with bounding boxes
img = cv2.imread(test_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convert to RGB format

for r in results:
    for box in r.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = box.conf[0]
        cls = int(box.cls[0])
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)  # Draw bounding boxes
        text = f"{model.names[cls]}: {conf:.2f}"
        cv2.putText(img, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.show()